# Step 4: Detect (yes/no)

![Step 4 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/04-step.png)

## What you're building today

The computer learns to ask one question: **is there a bird in this picture?**

If yes, it draws a box around the bird and tells you where. If no, it says "no bird".

By the end of this notebook, every frame will come with an answer: bird or no bird, and where.

## Step 4.1 — What is detection?

**Detection** means: look at a picture, find the things in it, and say WHERE each thing is.

We don't need to know what KIND of bird yet. Just: is one there?

We'll use **YOLO** ("You Only Look Once"). It's a fast, popular detection model that knows about 80 common things, including "bird". A version of it was trained on millions of labeled photos so it can spot things better than you or me.

YOLO returns a list of boxes: `[(x, y, width, height), ...]` — one per detected object, plus what it thinks it is.

In [1]:
# ultralytics gives us YOLO. OpenCV helps us draw on the image.
%pip install -q ultralytics opencv-python-headless
print("OK")

/home/jaewilson07/GitHub/bird-watcher/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
OK


## Step 4.2 — Load the model

The first run downloads the YOLO weights (~6MB). After that it's cached and instant.

We're using `yolov8n` — the smallest, fastest version. For a phone-camera frame, it's plenty.

In [5]:
VISION_MODEL = "yolov8n.pt"  # YOLOv8n model


In [ ]:
from ultralytics import YOLO

model = YOLO(VISION_MODEL)  # downloads on first run
print(f"Loaded {len(model.names)} classes")

bird_class_id = next((k for k, v in model.names.items() if v == 'bird'), None)

print(f"Class index for 'bird': {bird_class_id if bird_class_id is not None else 'not found'}")

Loaded 80 classes
Class index for 'bird': 14


## Step 4.3 — Grab a frame

Reuse the grab function from notebook 03.

In [4]:
import requests, os
from pathlib import Path
from PIL import Image
from IPython.display import display


PHONE_IP = "192.168.1.42"
SOURCE = f"http://{PHONE_IP}:8080/photo.jpg"

frame_path = Path("data/snapshots/detect-input.jpg")
frame_path.write_bytes(requests.get(SOURCE, timeout=10).content)
display(Image.open(frame_path))

ConnectionError: HTTPConnectionPool(host='192.168.1.42', port=8080): Max retries exceeded with url: /photo.jpg (Caused by NewConnectionError("HTTPConnection(host='192.168.1.42', port=8080): Failed to establish a new connection: [Errno 113] No route to host"))

## Step 4.4 — Run detection

YOLO returns a result object. For each detection, we get: class name, confidence (0-1), bounding box (xyxy coordinates).

In [ ]:
results = model(frame_path, verbose=False)
result = results[0]

# Filter to just birds (class 14 in COCO)
BIRD_CLASS_ID = 14
birds = [
    box for box in result.boxes
    if int(box.cls[0]) == BIRD_CLASS_ID and float(box.conf[0]) > 0.4
]

if not birds:
    print("no bird")
else:
    for box in birds:
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
        conf = float(box.conf[0])
        print(f"bird found at ({x1}, {y1}, {x2}, {y2})  conf={conf:.2f}")

## Step 4.5 — Draw the box on the image

Numbers are useful, but a picture with a box around the bird is way more satisfying.

In [ ]:
import cv2

img = cv2.imread(str(frame_path))
for box in birds:
    x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
    cv2.putText(img, f"bird {float(box.conf[0]):.2f}", (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

# Save and show
out = Path("data/snapshots/detect-output.jpg")
cv2.imwrite(str(out), img)
display(Image.open(out))

## Done?

If you saw either:
- `bird found at (x, y, x, y)` with a green box on the image, OR
- `no bird` printed cleanly

...then the brain is wired up. 🎉

## What's next

Issue **#5: Identify species**. We can spot a bird now, but we still don't know what KIND. Time to name names — Robin, Cardinal, Sparrow — with a species classifier. 🐦